# Kombinierte RBF-SVM: Spektrales Jittering

## Fragestellung

Die RBF-SVM auf standardisierten Rohspektren mit AEZ und Month ist die dokumentierte Referenz des kombinierten Ansatzes. Dieses Notebook prüft ausschließlich auf den fünf Trainingsfolds, ob kleine, label-erhaltende Störungen der Spektralbänder die kombinierte Balanced Accuracy verbessern.

Der gemeinsame Holdout wurde für die SVM-Familie bereits ausgewertet. Deshalb enthält dieses Notebook bewusst keinen `run()`-Aufruf und erzeugt keine neue Holdout-Metrik.

## Versuchsdesign

Die Referenz-SVM verwendet `C=100.0` und `gamma=0.003`. Die Kontrolle erhält keine zusätzlichen Spektren. Die beiden Augmentierungskandidaten erhalten pro Trainingsspektrum eine Kopie mit normalverteiltem Rauschen von `0.01` beziehungsweise `0.03` Standardabweichungen je standardisiertem Band. AEZ und Month bleiben unverändert.

Die Kopien entstehen ausschließlich im `fit()` des Klassifikators. Damit sehen die CV-Testfolds, die frühere Holdout-Menge und spätere Vorhersagen immer unveränderte Spektren.

In [ ]:
import importlib.metadata
import platform
import subprocess

from IPython.display import display

from awp2.config import SEED, SVM_REFERENCE_C, SVM_REFERENCE_GAMMA
from awp2.experiment import TuneConfig, tune
from awp2.models import make_combined_spectral_jitter_svm
from awp2.preprocessing import PreprocessingConfig

EXPERIMENT_ID = "combined_svm_spectral_jitter"
TUNING_ID = f"{EXPERIMENT_ID}_search"
PREPROCESSING = PreprocessingConfig(use_spectral_standard_scale=True)
PARAM_GRID = [
    {
        "estimator__augmentation_multiplier": [0],
        "estimator__noise_std": [0.0],
    },
    {
        "estimator__augmentation_multiplier": [1],
        "estimator__noise_std": [0.01, 0.03],
    },
]

environment = {
    "python": platform.python_version(),
    "scikit_learn": importlib.metadata.version("scikit-learn"),
    "branch": subprocess.check_output(
        ["git", "branch", "--show-current"],
        text=True,
    ).strip(),
    "random_state": SEED,
    "reference_svm": {"C": SVM_REFERENCE_C, "gamma": SVM_REFERENCE_GAMMA},
    "preprocessing": PREPROCESSING.model_dump(),
    "param_grid": PARAM_GRID,
}
display(environment)

tuned = tune(
    make_combined_spectral_jitter_svm(),
    TuneConfig(
        name=TUNING_ID,
        approach="combined",
        description=(
            "CV-only: Spektrales Jittering für die kombinierte RBF-SVM-Referenz "
            "auf standardisierten Rohspektren."
        ),
        preprocessing=PREPROCESSING,
        param_grid=PARAM_GRID,
    ),
)
results = tuned.candidates.sort_values("rank")
display(results)
{
    "best_cv_score": tuned.best_score,
    "best_params": tuned.best_params,
    "tuning_run": tuned.run_id,
}

## Auswertung

Vergleiche die Kontrolle mit den Jitter-Kandidaten anhand mittlerer CV-BAcc und Fold-Streuung. Ein Vorsprung unter `MIN_CV_GAIN = 0.005` gilt als Rauschen. Prüfe zusätzlich, ob stärkeres Rauschen die Streuung erhöht oder die mittlere Leistung senkt.

Auch bei einem klaren CV-Gewinn wird kein weiterer Holdout-Lauf gestartet: Der gemeinsame Holdout wurde für die SVM-Familie bereits verwendet.